Step 1: Preprocessing meteostat data

In [2]:
from meteostat import Point, Hourly
from datetime import datetime
import pandas as pd
import numpy as np

In [3]:
# Parameters
lat, lon = 34.0008, -81.0351
start = datetime(2024,5,26)
end = datetime(2024,5,27)
city = Point(lat, lon)

In [4]:
# Fetch
df = Hourly(city, start, end).fetch()
print(df)

# ensure timestamps are datetime
df.index = pd.to_datetime(df.index)

# Select & rename columns
df = df[['temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'pres', 'coco']]

# Ensure hourly continuity
full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='H', tz=df.index.tz)
df = df.reindex(full_idx)

                     temp  dwpt  rhum  prcp  snow   wdir  wspd  wpgt    pres  \
time                                                                           
2024-05-26 00:00:00  29.4  16.2  45.0   0.0  <NA>  250.0  13.0  <NA>  1010.6   
2024-05-26 01:00:00  26.7  17.7  58.0   0.0  <NA>  230.0   5.4  <NA>  1011.1   
2024-05-26 02:00:00  26.7  17.7  58.0   0.0  <NA>  270.0  11.2  <NA>  1012.0   
2024-05-26 03:00:00  25.6  17.8  62.0   0.0  <NA>  102.0  11.2  <NA>  1012.6   
2024-05-26 04:00:00  23.9  18.3  71.0   0.0  <NA>    0.0   0.0  <NA>  1012.5   
2024-05-26 05:00:00  22.8  18.3  76.0   0.0  <NA>    0.0   0.0  <NA>  1012.6   
2024-05-26 06:00:00  21.7  18.3  81.0   0.0  <NA>  250.0   5.4  <NA>  1012.1   
2024-05-26 07:00:00  21.1  18.3  84.0   0.0  <NA>    0.0   0.0  <NA>  1011.5   
2024-05-26 08:00:00  20.6  18.4  87.0   0.0  <NA>    0.0   0.0  <NA>  1011.5   
2024-05-26 09:00:00  20.0  18.3  90.0   0.0  <NA>    0.0   0.0  <NA>  1011.2   
2024-05-26 10:00:00  19.4  17.7  90.0   

In [5]:
# Change precip to either 0 or log-transformed amount
df['prcp'] = np.log1p(df['prcp'])
# add a rain flag (0 or 1)
df['prcp_flag'] = (df['prcp'] > 0).astype(int)
print(df)

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-05-26 00:00:00  29.4  16.2  45.0   0.0  250.0  13.0  1010.6   3.0   
2024-05-26 01:00:00  26.7  17.7  58.0   0.0  230.0   5.4  1011.1  17.0   
2024-05-26 02:00:00  26.7  17.7  58.0   0.0  270.0  11.2  1012.0  17.0   
2024-05-26 03:00:00  25.6  17.8  62.0   0.0  102.0  11.2  1012.6   1.0   
2024-05-26 04:00:00  23.9  18.3  71.0   0.0    0.0   0.0  1012.5   1.0   
2024-05-26 05:00:00  22.8  18.3  76.0   0.0    0.0   0.0  1012.6   1.0   
2024-05-26 06:00:00  21.7  18.3  81.0   0.0  250.0   5.4  1012.1   1.0   
2024-05-26 07:00:00  21.1  18.3  84.0   0.0    0.0   0.0  1011.5   1.0   
2024-05-26 08:00:00  20.6  18.4  87.0   0.0    0.0   0.0  1011.5   1.0   
2024-05-26 09:00:00  20.0  18.3  90.0   0.0    0.0   0.0  1011.2   1.0   
2024-05-26 10:00:00  19.4  17.7  90.0   0.0    0.0   0.0  1011.7   1.0   
2024-05-26 11:00:00  19.4  18.2  93.0   0.0    0.0   0.0  1012.8   3.0   
2024-05-26 12:00:00  21.1  19.4  90.0 

In [6]:
# Impute missing timestamps
# short gaps (<=3h): linear; long gaps: forward fill and add mask
gap_mask = df.isna().any(axis=1)
#print(gap_mask[1].sum())
df_short = df.interpolate(limit=3, limit_direction='both')
print(df_short)
df_long = df_short.fillna(method='ffill')
print(df_long)

                     temp  dwpt  rhum  prcp   wdir  wspd    pres  coco  \
2024-05-26 00:00:00  29.4  16.2  45.0   0.0  250.0  13.0  1010.6   3.0   
2024-05-26 01:00:00  26.7  17.7  58.0   0.0  230.0   5.4  1011.1  17.0   
2024-05-26 02:00:00  26.7  17.7  58.0   0.0  270.0  11.2  1012.0  17.0   
2024-05-26 03:00:00  25.6  17.8  62.0   0.0  102.0  11.2  1012.6   1.0   
2024-05-26 04:00:00  23.9  18.3  71.0   0.0    0.0   0.0  1012.5   1.0   
2024-05-26 05:00:00  22.8  18.3  76.0   0.0    0.0   0.0  1012.6   1.0   
2024-05-26 06:00:00  21.7  18.3  81.0   0.0  250.0   5.4  1012.1   1.0   
2024-05-26 07:00:00  21.1  18.3  84.0   0.0    0.0   0.0  1011.5   1.0   
2024-05-26 08:00:00  20.6  18.4  87.0   0.0    0.0   0.0  1011.5   1.0   
2024-05-26 09:00:00  20.0  18.3  90.0   0.0    0.0   0.0  1011.2   1.0   
2024-05-26 10:00:00  19.4  17.7  90.0   0.0    0.0   0.0  1011.7   1.0   
2024-05-26 11:00:00  19.4  18.2  93.0   0.0    0.0   0.0  1012.8   3.0   
2024-05-26 12:00:00  21.1  19.4  90.0 

In [7]:
# Standardize numerical units and clip outliers (3 sigma)
df_std= df_long.copy()
for col in ['temp', 'dwpt', 'rhum', 'wdir', 'wspd', 'pres']:
    var = df_long[col]
    mu, sigma = var.mean(), var.std()
    # clip values greater than 3 stds
    clipped = var.clip(lower=mu - 3*sigma, upper=mu + 3*sigma)
    # standardize by subtracting mean then dividing by std
    df_long[col] = (clipped - clipped.mean()) / clipped.std()

print(df_long)

                         temp      dwpt      rhum  prcp      wdir      wspd  \
2024-05-26 00:00:00    0.5253 -1.901935  -0.90083   0.0  0.971023  1.478254   
2024-05-26 01:00:00 -0.002345 -0.458158 -0.251673   0.0  0.785713 -0.092598   
2024-05-26 02:00:00 -0.002345 -0.458158 -0.251673   0.0  1.156332   1.10621   
2024-05-26 03:00:00 -0.217312 -0.361907 -0.051933   0.0 -0.400269   1.10621   
2024-05-26 04:00:00 -0.549533  0.119352  0.397484   0.0 -1.345348 -1.208729   
2024-05-26 05:00:00   -0.7645  0.119352  0.647159   0.0 -1.345348 -1.208729   
2024-05-26 06:00:00 -0.979466  0.119352  0.896835   0.0  0.971023 -0.092598   
2024-05-26 07:00:00 -1.096721  0.119352   1.04664   0.0 -1.345348 -1.208729   
2024-05-26 08:00:00 -1.194433  0.215604  1.196446   0.0 -1.345348 -1.208729   
2024-05-26 09:00:00 -1.311687  0.119352  1.346251   0.0 -1.345348 -1.208729   
2024-05-26 10:00:00 -1.428942 -0.458158  1.346251   0.0 -1.345348 -1.208729   
2024-05-26 11:00:00 -1.428942    0.0231  1.496056   

In [8]:
# Save cleaned dataset
df_long.to_parquet('city_weather_clean.parquet')